# 03 — Özellik Seçimi

**Amaç:** 3 farklı özellik seçimi stratejisini karşılaştırmak:
1. Korelasyon analizi
2. Random Forest önem skorları (Top-20)
3. Mutual Information (k=20)
4. Vajrobol'un 5 özelliği (literatür referansı)

**Not:** Bu notebook'un sonunda bulunan `feature_names_*` değişkenleri,
04_model_training.ipynb'de kullanılacaktır.

## 0. Colab Kurulumu

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/phishing-url-detection'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/erenterakye/phishing-url-detection.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull

    %cd {REPO_DIR}
    !pip install -q -r requirements.txt

    DATA_PATH = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'
else:
    REPO_DIR = '..'
    DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'

sys.path.insert(0, REPO_DIR)
print(f'IN_COLAB={IN_COLAB}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import full_preprocessing_pipeline
from src.feature_selection import (
    correlation_analysis, rf_feature_importance,
    mutual_info_selection, get_top5_vajrobol
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Ön İşlenmiş Veri

In [ ]:
X_train, X_test, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)
feature_names = list(X_train.columns)
print(f'Özellik sayısı: {len(feature_names)}')

## 2. Korelasyon Analizi

In [ ]:
high_corr_features, _ = correlation_analysis(X_train, threshold=0.9)
print(f'Yüksek korelasyonlu (>0.9) özellik sayısı: {len(high_corr_features)}')

## 3. Random Forest Önem Skorları (birkaç dakika sürer)

In [ ]:
print('RF önem skorları hesaplanıyor...')
rf_top20, rf_importance_df = rf_feature_importance(X_train, y_train, top_n=20)

fig, ax = plt.subplots(figsize=(10, 7))
top20_df = rf_importance_df.head(20)
ax.barh(range(20), top20_df['importance'].values[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(top20_df['feature'].values[::-1], fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('RF Feature Importance — Top-20')
plt.tight_layout()
plt.show()

print('\nTop-20 RF özellikleri:')
print(rf_top20)

## 4. Mutual Information Seçimi

In [ ]:
print('Mutual Information hesaplanıyor...')
mi_top20, mi_scores = mutual_info_selection(X_train, y_train, k=20)

fig, ax = plt.subplots(figsize=(10, 7))
mi_top20_df = mi_scores.head(20)
ax.barh(range(20), mi_top20_df['mi_score'].values[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(mi_top20_df['feature'].values[::-1], fontsize=9)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information — Top-20 Özellik')
plt.tight_layout()
plt.show()

## 5. Vajrobol'un 5 Özelliği

In [ ]:
vajrobol_5 = [f for f in get_top5_vajrobol() if f in feature_names]
print('Vajrobol\'un 5 özelliği:', vajrobol_5)

## 6. Karşılaştırma

In [ ]:
rf_set = set(rf_top20)
mi_set = set(mi_top20)
vaj_set = set(vajrobol_5)

comparison = pd.DataFrame({
    'Feature': sorted(rf_set | mi_set | vaj_set),
})
comparison['RF Top-20'] = comparison['Feature'].isin(rf_set)
comparison['MI Top-20'] = comparison['Feature'].isin(mi_set)
comparison['Vajrobol']  = comparison['Feature'].isin(vaj_set)

print(comparison.to_string(index=False))
print(f'\nRF ∩ MI ortak özellik sayısı: {len(rf_set & mi_set)}')
print(f'Vajrobol ⊂ RF Top-20: {vaj_set.issubset(rf_set)}')
print(f'Vajrobol ⊂ MI Top-20: {vaj_set.issubset(mi_set)}')

## 7. Sonuç — Bir Sonraki Notebook İçin Özellik Setleri

Aşağıdaki değişkenler `04_model_training.ipynb`'de kullanılacak:
- `feature_names` → Tüm özellikler
- `rf_top20` → RF Top-20
- `vajrobol_5` → Vajrobol'un 5 özelliği

In [ ]:
print('=== Özellik Seti Özeti ===')
print(f'Tüm özellikler : {len(feature_names)}')
print(f'RF Top-20      : {len(rf_top20)}')
print(f'MI Top-20      : {len(mi_top20)}')
print(f'Vajrobol-5     : {len(vajrobol_5)}')